# Time-Walk Calibration & Alignment Parameters Analysis

This notebook provides a comprehensive study of the Time-Walk calibration and fine alignment constants ($d_{\mathrm{SiPM}}$) extracted from Category II runs:
1. **Data Ingestion & Loaders:** Ingests all run calibrations from `/eos/user/i/idioniso/Timing/timewalk/fits` and provides ROOT data loading utilities for 2D histograms and residuals from `/eos/user/i/idioniso/Timing/timewalk/runs`.
2. **Evolution Across Time (Run Number):** Studies the run-level drift and stability of all parameters ($d_{\mathrm{SiPM}}$, $t_0$, $\alpha$, $\beta$, $\gamma$, and $\sigma_{\mathrm{sys}}$).
3. **Individual Channel Stability:** Tracks representative channels across planes and subsystems vs. the global run median.
4. **Full Detector Alignment Map (Veto + US):** Replicates the thesis alignment parameter distribution (Figure 4.38) as a function of SiPM number, including gel side indicators ("G") and plane boundaries for both Veto and Upstream systems.
5. **Raw & Time-Walk Corrected 2D Distributions:** Directly illustrates the uncorrected raw time-walk curve, the fitted rational function, the corrected flattened plateau, and the fine alignment parameter ($d_{\mathrm{SiPM}}$) extraction with $\pm 1\sigma$ truncation window.


In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats
import concurrent.futures

try:
    import ddfUtils.mpl_styling as dms
    has_dms = True
except ImportError:
    has_dms = False

plt.rcParams.update({
    "font.size": 13,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "figure.titlesize": 15,
    "figure.autolayout": True
})
print("Environment initialized successfully.")


---
## 1. Data Ingestion & Loading Utilities

Loads all calibration files into tabular datasets (`df_channels`, `df_runs`, `cat_al`), and defines `load_channel_tw_root(...)` to ingest 2D time-walk distributions, fit parameters, and residual projections directly from ROOT files.


In [ ]:
fits_dir = "/eos/user/i/idioniso/Timing/timewalk/fits"
runs_dir = "/eos/user/i/idioniso/Timing/timewalk/runs"
json_files = sorted(glob.glob(os.path.join(fits_dir, "run*_timing_calibration.json")))

def get_run_period(run: int) -> str:
    """Classifies run into data-taking periods."""
    if 5000 <= run < 6000:
        return "Cat II (2022)"
    elif 6000 <= run < 7000:
        return "Cat III (2023)"
    elif 10000 <= run < 13000:
        return "Cat III (2025)"
    elif 13000 <= run < 14000:
        return "Cat III (2026)"
    return "Other"

# 1. Ingest Category II consolidated calibration (if available)
cat_file = os.path.join(fits_dir, "catII_timing_calibration.json")
has_cat_file = os.path.exists(cat_file)
cat_al = {}
if has_cat_file:
    with open(cat_file, "r") as f:
        cat_data = json.load(f)
    cat_al = cat_data.get("alignment", {})
    print(f"Loaded consolidated Category II calibration ({len(cat_al.get('DS', {}))} DS, {len(cat_al.get('SF', {}))} SF channels).")

# 2. Ingest run-by-run calibration files with parallel reading
def parse_single_calib(jf):
    with open(jf, "r") as f:
        data = json.load(f)
    r = data.get("run", 0)
    cat = data.get("category", "catII")
    period = get_run_period(r)
    al_all = data.get("alignment", {})
    tw_all = data.get("timewalk", {})

    ch_records = []
    run_records = []

    for ref in ["DS", "SF"]:
        al_dict = al_all.get(ref, {})
        tw_dict = tw_all.get(ref, {})
        if not al_dict:
            continue

        d_vals, t0_vals, a_vals, b_vals = [], [], [], []
        q_vals, g_vals, s_vals, c_vals, cs_vals = [], [], [], [], []

        for ch_key, al_info in al_dict.items():
            det_id, sipm = [int(p) for p in ch_key.split("_")]
            subsys = det_id // 10000
            plane = (det_id % 10000) // 1000
            bar = det_id % 1000
            side = "left" if sipm < 8 else "right"
            system_name = "Veto" if subsys == 1 else ("US" if subsys == 2 else "DS")
            plane_label = f"{system_name} Plane {plane + 1}"

            d_val = al_info[0]
            d_err = al_info[1] if len(al_info) > 1 else 0.0
            tw_p = tw_dict.get(ch_key, {})

            d_vals.append(d_val)
            if "t0" in tw_p and tw_p["t0"] is not None: t0_vals.append(tw_p["t0"])
            if "alpha" in tw_p and tw_p["alpha"] is not None: a_vals.append(tw_p["alpha"])
            if "beta" in tw_p and tw_p["beta"] is not None: b_vals.append(tw_p["beta"])
            if "qdc0" in tw_p and tw_p["qdc0"] is not None: q_vals.append(tw_p["qdc0"])
            if "gamma" in tw_p and tw_p["gamma"] is not None: g_vals.append(tw_p["gamma"])
            if "sigma_sys" in tw_p and tw_p["sigma_sys"] is not None: s_vals.append(tw_p["sigma_sys"])
            if "chi2_tot_ndf" in tw_p and tw_p["chi2_tot_ndf"] is not None: c_vals.append(tw_p["chi2_tot_ndf"])
            if "chi2_stat_ndf" in tw_p and tw_p["chi2_stat_ndf"] is not None: cs_vals.append(tw_p["chi2_stat_ndf"])

            ch_records.append({
                "run": r,
                "category": cat,
                "period": period,
                "ref": ref,
                "channel": ch_key,
                "detID": det_id,
                "subsys": subsys,
                "system": system_name,
                "plane": plane,
                "plane_label": plane_label,
                "bar": bar,
                "sipm": sipm,
                "side": side,
                "d_sipm": d_val,
                "d_sipm_err": d_err,
                "t0": tw_p.get("t0", np.nan),
                "alpha": tw_p.get("alpha", np.nan),
                "beta": tw_p.get("beta", np.nan),
                "qdc0": tw_p.get("qdc0", np.nan),
                "gamma": tw_p.get("gamma", np.nan),
                "sigma_sys": tw_p.get("sigma_sys", np.nan),
                "chi2_tot_ndf": tw_p.get("chi2_tot_ndf", np.nan),
                "chi2_stat_ndf": tw_p.get("chi2_stat_ndf", np.nan),
                "rms_residual": tw_p.get("rms_residual", np.nan),
            })

        def calc_stat(vals):
            if not vals:
                return np.nan, np.nan
            arr = np.array(vals)
            med = float(np.median(arr))
            sem = float(1.253 * np.std(arr, ddof=1) / np.sqrt(len(arr))) if len(arr) > 1 else 0.0
            return med, sem

        d_med, d_sem = calc_stat(d_vals)
        t0_med, t0_sem = calc_stat(t0_vals)
        a_med, a_sem = calc_stat(a_vals)
        b_med, b_sem = calc_stat(b_vals)
        q_med, q_sem = calc_stat(q_vals)
        g_med, g_sem = calc_stat(g_vals)
        s_med, s_sem = calc_stat(s_vals)
        c_med, c_sem = calc_stat(c_vals)
        cs_med, cs_sem = calc_stat(cs_vals)

        run_records.append({
            "run": r,
            "category": cat,
            "period": period,
            "ref": ref,
            "n_channels": len(d_vals),
            "d_median": d_med, "d_sem": d_sem,
            "t0_median": t0_med, "t0_sem": t0_sem,
            "alpha_median": a_med, "alpha_sem": a_sem,
            "beta_median": b_med, "beta_sem": b_sem,
            "qdc0_median": q_med, "qdc0_sem": q_sem,
            "gamma_median": g_med, "gamma_sem": g_sem,
            "sigma_sys_median": s_med, "sigma_sys_sem": s_sem,
            "chi2_tot_median": c_med, "chi2_tot_sem": c_sem,
            "chi2_stat_median": cs_med, "chi2_stat_sem": cs_sem,
        })
    return ch_records, run_records

with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    parsed_results = list(executor.map(parse_single_calib, json_files))

channel_records = [rec for p in parsed_results for rec in p[0]]
run_summary_records = [rec for p in parsed_results for rec in p[1]]

df_channels = pd.DataFrame(channel_records)
df_runs = pd.DataFrame(run_summary_records)
print(f"Data ingestion complete:")
print(f"  - Total channel records: {len(df_channels)}")
print(f"  - Total run summaries:   {len(df_runs)}")
print(f"  - Runs per period:       {dict(df_runs[df_runs['ref'] == 'DS']['period'].value_counts())}")

# 3. Helper to load full 2D root histograms for single channels
def load_channel_tw_root(sample_run: int, target_ch: str, ref_type: str = "DS",
                         y_min: float = -20.0, y_max: float = 20.0, n_y_bins: int = 200) -> dict:
    """Loads 2D dt vs QDC histogram and 1D residual distribution from the merged run ROOT file."""
    import ROOT
    ROOT.gROOT.SetBatch(True)
    
    root_file_path = os.path.join(runs_dir, f"run{sample_run}.root")
    json_file_path = os.path.join(fits_dir, f"run{sample_run}_timing_calibration.json")
    
    if not os.path.exists(json_file_path):
        raise FileNotFoundError(f"Cannot open {json_file_path}")
    with open(json_file_path, "r") as f:
        calib_data = json.load(f)
    tw_params = calib_data.get("timewalk", {}).get(ref_type, {}).get(target_ch, {})
    if not tw_params:
        raise KeyError(f"Channel {target_ch} ({ref_type}) not found in {json_file_path}")
    
    rf = ROOT.TFile.Open(root_file_path, "READ")
    if not rf or rf.IsZombie():
        raise FileNotFoundError(f"Cannot open {root_file_path}")
    
    h2 = rf.Get(f"{target_ch}/dtvqdc-{ref_type}")
    h_res = rf.Get(f"{target_ch}/dt_residual_{ref_type}_{target_ch}")
    if not h2:
        rf.Close()
        raise KeyError(f"Histogram {target_ch}/dtvqdc-{ref_type} not found in {root_file_path}")
    
    nbins_x = h2.GetNbinsX()
    nbins_y = h2.GetNbinsY()
    x_min, x_max = 0.0, 200.0
    x_edges = np.linspace(x_min, x_max, nbins_x + 1)
    y_edges = np.linspace(y_min, y_max, n_y_bins + 1)
    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    
    alpha = tw_params.get("alpha", 0.0)
    beta = tw_params.get("beta", 1.0)
    qdc0 = tw_params.get("qdc0", 0.0)
    gamma = tw_params.get("gamma", 0.0)
    tw_corr = (alpha / (beta * x_centers - qdc0)) + gamma * x_centers
    
    raw_2d = np.zeros((nbins_y, nbins_x), dtype=np.float64)
    corr_2d = np.zeros((n_y_bins, nbins_x), dtype=np.float64)
    
    for bx in range(1, nbins_x + 1):
        shift = tw_corr[bx - 1]
        for by in range(1, nbins_y + 1):
            cnt = h2.GetBinContent(bx, by)
            if cnt <= 0: continue
            raw_2d[by - 1, bx - 1] = cnt
            dt_val = h2.GetYaxis().GetBinCenter(by)
            dt_tw = dt_val - shift
            if y_min <= dt_tw <= y_max:
                by_idx = int((dt_tw - y_min) / ((y_max - y_min) / n_y_bins))
                if 0 <= by_idx < n_y_bins:
                    corr_2d[by_idx, bx - 1] += cnt
                    
    raw_y_edges = np.array([h2.GetYaxis().GetBinLowEdge(i) for i in range(1, nbins_y + 2)])
    
    res_x, res_y = [], []
    sigma_dist = h_res.GetStdDev() if h_res else np.nan
    if h_res:
        for b in range(1, h_res.GetNbinsX() + 1):
            xc = h_res.GetBinCenter(b)
            if y_min <= xc <= y_max:
                res_x.append(xc)
                res_y.append(h_res.GetBinContent(b))
    rf.Close()
    
    return {
        "run": sample_run,
        "channel": target_ch,
        "ref": ref_type,
        "params": tw_params,
        "x_edges": x_edges,
        "x_centers": x_centers,
        "y_edges": y_edges,
        "raw_y_edges": raw_y_edges,
        "raw_2d": raw_2d,
        "corr_2d": corr_2d,
        "tw_corr": tw_corr,
        "res_x": np.array(res_x),
        "res_y": np.array(res_y),
        "sigma_dist": sigma_dist,
    }

print("Data loading utility load_channel_tw_root defined successfully.")


---
## 2. Parameter Stability Across Time (Run Number)

Visualizes the run-by-run evolution of all Time-Walk and Alignment parameters across the detector:
- $t_0$ [ns]: Rational offset / reference time shift
- $\alpha$: Scale parameter
- $\beta$: Curvature parameter
- $\mathrm{QDC}_0$: Charge threshold / horizontal offset
- $\gamma$: Linear asymptotic slope
- $\sigma_{\mathrm{sys}}$ [ns]: Systematic error floor
- $\chi^2_{\mathrm{tot}} / \mathrm{ndf}$: Fit quality
- $d_{\mathrm{SiPM}}$ [ns]: Fine alignment parameter
- Number of Fitted Channels ($N_{\mathrm{channels}}$)

Results are separated into dedicated subfigures for each data-taking period:
1. **Category II (2022 Runs)**: 33 runs (5119–5408)
2. **Category III (2023 Runs)**: 10 runs (6079–6667)
3. **Category III (2025 Runs)**: 10 runs (11124–12217)
4. **Category III (2026 Runs)**: 10 runs (13085–13520)


In [ ]:
def plot_tw_parameter_stability(df_runs, period_name, target=None, show_median_line=True):
    """
    Plots the run-by-run stability of all Time-Walk and Alignment parameters for a given period.
    Parameters plotted: t0, alpha, beta, QDC0, gamma, sigma_sys, chi2_tot/ndf, d_SiPM, and channel count.
    Supports either standalone figure creation or rendering into a matplotlib SubFigure.
    """
    df_period = df_runs[df_runs["period"] == period_name].copy()
    if df_period.empty:
        print(f"No run records found for period: {period_name}")
        return None

    is_standalone = (target is None)
    if is_standalone:
        fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True)
    else:
        fig = target
        axes = target.subplots(3, 3, sharex=True)

    params = [
        ("t0_median", "t0_sem", r"$t_0$ [ns]", "Rational Offset ($t_0$)"),
        ("alpha_median", "alpha_sem", r"$\alpha$", r"Scale Parameter ($\alpha$)"),
        ("beta_median", "beta_sem", r"$\beta$", r"Curvature Parameter ($\beta$)"),
        ("qdc0_median", "qdc0_sem", r"$\mathrm{QDC}_0$", r"Charge Offset ($\mathrm{QDC}_0$)"),
        ("gamma_median", "gamma_sem", r"$\gamma$", r"Linear Slope ($\gamma$)"),
        ("sigma_sys_median", "sigma_sys_sem", r"$\sigma_{\mathrm{sys}}$ [ns]", r"Systematic Jitter ($\sigma_{\mathrm{sys}}$)"),
        ("chi2_tot_median", "chi2_tot_sem", r"$\chi^2_{\mathrm{tot}} / \mathrm{ndf}$", r"Fit Quality ($\chi^2_{\mathrm{tot}}/\mathrm{ndf}$)"),
        ("d_median", "d_sem", r"$d_{\mathrm{SiPM}}$ [ns]", r"Fine Alignment ($d_{\mathrm{SiPM}}$)"),
        ("n_channels", None, "Fitted Channels", "Channel Multiplicity"),
    ]

    for idx, (med_col, sem_col, y_lbl, p_title) in enumerate(params):
        ax = axes.flatten()[idx]
        for ref, color, marker in zip(["DS", "SF"], ["steelblue", "crimson"], ["o", "s"]):
            df_sub = df_period[df_period["ref"] == ref].sort_values("run")
            if df_sub.empty:
                continue
            if sem_col is not None:
                ax.errorbar(
                    df_sub["run"],
                    df_sub[med_col],
                    yerr=df_sub[sem_col],
                    marker=marker,
                    markersize=5,
                    linewidth=1.2,
                    capsize=2,
                    color=color,
                    label=f"{ref} Ref (Median ± SEM)"
                )
                if show_median_line:
                    cat_median = df_sub[med_col].median()
                    ax.axhline(cat_median, color=color, linestyle=":", alpha=0.6)
            else:
                ax.plot(
                    df_sub["run"],
                    df_sub[med_col],
                    marker=marker,
                    markersize=5,
                    linewidth=1.3,
                    color=color,
                    label=f"{ref} Ref"
                )
        ax.set_ylabel(y_lbl)
        ax.set_title(p_title, fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.legend(frameon=True, fontsize=8, loc="best")

    for ax in axes[-1, :]:
        ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%d"))
        ax.set_xlabel("Run Number")
        ax.tick_params(axis="x", rotation=30)

    title_str = f"SND@LHC — Time-Walk & Alignment Parameters Stability: {period_name}"
    if is_standalone:
        fig.suptitle(title_str, y=0.995, fontsize=15, fontweight="bold")
        fig.tight_layout()
    else:
        fig.suptitle(f"{period_name}", fontsize=13, fontweight="bold")

    return fig


### 2.1 Category II (2022 Runs)

Plots parameter trends for **Cat II (2022)** runs.


In [ ]:
fig_catII_2022 = plot_tw_parameter_stability(df_runs, "Cat II (2022)")
plt.show()


### 2.2 Category III (2023 Runs)

Plots parameter trends for **Cat III (2023)** runs.


In [ ]:
fig_catIII_2023 = plot_tw_parameter_stability(df_runs, "Cat III (2023)")
plt.show()


### 2.3 Category III (2025 Runs)

Plots parameter trends for **Cat III (2025)** runs.


In [ ]:
fig_catIII_2025 = plot_tw_parameter_stability(df_runs, "Cat III (2025)")
plt.show()


### 2.4 Category III (2026 Runs)

Plots parameter trends for **Cat III (2026)** runs.


In [ ]:
fig_catIII_2026 = plot_tw_parameter_stability(df_runs, "Cat III (2026)")
plt.show()


### 2.5 Combined Multi-Period Subfigures Canvas

Arranges all four periods into a single 4-quadrant layout using `matplotlib.figure.SubFigure`.


In [ ]:
fig_all = plt.figure(figsize=(26, 22))
subfigs = fig_all.subfigures(2, 2)
all_periods = ["Cat II (2022)", "Cat III (2023)", "Cat III (2025)", "Cat III (2026)"]
for sf, p in zip(subfigs.flatten(), all_periods):
    plot_tw_parameter_stability(df_runs, p, target=sf, show_median_line=True)
fig_all.suptitle("SND@LHC — Time-Walk Parameter Stability Comparison Across All Data Periods", fontsize=18, fontweight="bold", y=0.998)
plt.show()


---
## 3. Individual Channel Variations Across Time

Tracks representative individual channels across different planes and subsystems vs. the global run median (similar to the individual channel inspection in `cscint.ipynb`).


In [ ]:
representative_channels = [
    ("10003_3",  "Veto Pl 1, Bar 3, Left SiPM (DS ref)",  "#9467bd", "DS"),
    ("20004_4",  "US Pl 1, Bar 4, Left SiPM (DS ref)",    "#1f77b4", "DS"),
    ("20004_12", "US Pl 1, Bar 4, Right SiPM (DS ref)",   "#ff7f0e", "DS"),
    ("22005_11", "US Pl 3, Bar 5, Right SiPM (DS ref)",   "#2ca02c", "DS"),
    ("24000_4",  "US Pl 5, Bar 0, Left SiPM (DS ref)",    "#d62728", "DS"),
]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True, gridspec_kw={"height_ratios": [2.5, 1.2]})

# Global DS median baseline
df_ds_runs = df_runs[df_runs["ref"] == "DS"].sort_values("run")
ax1.plot(df_ds_runs["run"], df_ds_runs["d_median"], color="black", linewidth=2.5, linestyle="--", label="Global DS Median")

for ch_key, label_str, ch_color, ref in representative_channels:
    df_ch = df_channels[(df_channels["channel"] == ch_key) & (df_channels["ref"] == ref)].sort_values("run")
    if df_ch.empty:
        continue
    ax1.errorbar(df_ch["run"], df_ch["d_sipm"], yerr=df_ch["d_sipm_err"], fmt="-o", color=ch_color, markersize=5, label=label_str)
    # Residual to median in bottom panel
    merged = pd.merge(df_ch[["run", "d_sipm"]], df_ds_runs[["run", "d_median"]], on="run")
    ax2.plot(merged["run"], merged["d_sipm"] - merged["d_median"], marker="o", markersize=4, color=ch_color, label=ch_key)

ax1.set_ylabel(r"$d_{\mathrm{SiPM}}$ [ns]")
ax1.set_title("Individual Channel $d_{\mathrm{SiPM}}$ Stability vs. Global Median")
ax1.grid(True, alpha=0.3)
ax1.legend(loc="upper right", frameon=True, fontsize=10)

ax2.set_xlabel("Run Number")
ax2.set_ylabel(r"$\Delta d_{\mathrm{SiPM}}$ to Median [ns]")
ax2.axhline(0, color="black", linestyle=":", linewidth=1)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## 4. Full Detector Alignment Map ($d_{\mathrm{SiPM}}$ vs. SiPM Number)
### Including Both Veto and Upstream Systems

Replicates the structure of **Thesis Figure 4.38** with:
- Consecutive SiPM numbering for **Veto** (Planes 1–2, 112 channels/plane = 224 channels) and **Upstream** (Planes 1–5, 160 channels/plane = 800 channels).
- **Green solid lines** delineating detector planes.
- **Magenta dashed lines** separating **Left** (red points) and **Right** (blue points) readout sides.
- **"G" labels** indicating the SilGel side on each plane (matching Table 3.1 & Figure 4.38).


In [ ]:
# Select representative run (e.g. 5408 or 5119)
plot_run = 5408 if 5408 in df_channels["run"].values else df_channels["run"].iloc[0]
df_run = df_channels[(df_channels["run"] == plot_run) & (df_channels["ref"] == "DS")].copy()

# Build contiguous SiPM indices matching the layout:
# 1. Veto: Plane 0 (detID 10000..10006) and Plane 1 (11000..11006)
#    7 bars * 8 channels = 56 left + 56 right = 112 channels / plane
# 2. US: Planes 0..4 (detID 20000..24009)
#    10 bars * 8 channels = 80 left + 80 right = 160 channels / plane

def compute_full_sipm_index(row):
    subsys = row["subsys"]
    plane = row["plane"]
    bar = row["bar"]
    sipm = row["sipm"]
    is_left = (sipm < 8)
    sipm_in_side = sipm if is_left else (sipm - 8)
    if subsys == 1:  # Veto: offset 0 to 223
        plane_offset = plane * 112
        side_offset = 0 if is_left else 56
        return plane_offset + side_offset + bar * 8 + sipm_in_side
    elif subsys == 2:  # US: offset 224 to 1023
        us_base = 224
        plane_offset = plane * 160
        side_offset = 0 if is_left else 80
        return us_base + plane_offset + side_offset + bar * 8 + sipm_in_side
    return np.nan

df_run["global_sipm_idx"] = df_run.apply(compute_full_sipm_index, axis=1)
df_run = df_run.dropna(subset=["global_sipm_idx"]).sort_values("global_sipm_idx")

# Define plane boundaries & gel indicators
# Veto plane 0: [0, 112), left [0, 56), right [56, 112)
# Veto plane 1: [112, 224), left [112, 168), right [168, 224)
# US planes 0..4: each 160 ch
plane_boundaries = [0, 112, 224, 384, 544, 704, 864, 1024]
side_dividers = [56, 168, 304, 464, 624, 784, 944]

# Gel positions (Table 3.1 & Figure 4.38):
# US: Pl 0: Right, Pl 1: Left, Pl 2: Right, Pl 3: Left, Pl 4: Left
gel_annotations = [
    (84, "G"),   # Veto Pl 1 Right (illustrative)
    (196, "G"),  # Veto Pl 2 Right
    (344, "G"),  # US Pl 1 Right
    (424, "G"),  # US Pl 2 Left
    (664, "G"),  # US Pl 3 Right
    (744, "G"),  # US Pl 4 Left
    (904, "G"),  # US Pl 5 Left
]

fig, ax = plt.subplots(figsize=(16, 7))

# Split Left and Right points
df_left = df_run[df_run["side"] == "left"]
df_right = df_run[df_run["side"] == "right"]

ax.errorbar(df_left["global_sipm_idx"], df_left["d_sipm"], yerr=df_left["d_sipm_err"],
            fmt="+", color="red", markersize=6, capsize=0, label="left")
ax.errorbar(df_right["global_sipm_idx"], df_right["d_sipm"], yerr=df_right["d_sipm_err"],
            fmt="+", color="blue", markersize=6, capsize=0, label="right")

# Draw boundaries
for b in plane_boundaries:
    ax.axvline(b, color="lime", linewidth=2.0, linestyle="-")
for d in side_dividers:
    ax.axvline(d, color="magenta", linewidth=1.5, linestyle="--")

# Mark Gel side
y_top = ax.get_ylim()[1] if ax.get_ylim()[1] > 14 else 14.8
for x_pos, text in gel_annotations:
    ax.text(x_pos, y_top - 0.7, text, fontweight="bold", fontsize=13, ha="center", va="center")

# Label system regions
ax.text(112, y_top + 0.8, "VETO (Planes 1–2)", fontsize=12, fontweight="bold", ha="center",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.5))
ax.text(624, y_top + 0.8, "UPSTREAM (Planes 1–5)", fontsize=12, fontweight="bold", ha="center",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.5))

ax.set_xlim(-10, 1034)
ax.set_xlabel("SiPM number (Veto: 0–223 | Upstream: 224–1023)", fontsize=14)
ax.set_ylabel(r"DS alignment parameter, $d_{\mathrm{SiPM}}$ [ns]", fontsize=14)
ax.set_title(f"DS alignment parameter, {{\mathrm{{SiPM}}}}$ as a function of SiPM number (Run {plot_run})", fontsize=15, pad=25)
ax.grid(True, which="both", linestyle=":", alpha=0.5)
ax.legend(loc="upper right", frameon=True, framealpha=0.9, facecolor="white", edgecolor="black", fontsize=13)

plt.tight_layout()
plt.show()


---
## 5. Standalone Upstream Alignment Map (Exact Replica of Thesis Figure 4.38)

Isolated view of Upstream SiPMs (indices 0 to 799) with exact 160-channel plane groupings, gel indicators, and side dividers.


In [ ]:
df_us = df_run[df_run["subsys"] == 2].copy()
df_us["us_sipm_idx"] = df_us["global_sipm_idx"] - 224

fig, ax = plt.subplots(figsize=(15, 6))
df_us_left = df_us[df_us["side"] == "left"]
df_us_right = df_us[df_us["side"] == "right"]

ax.errorbar(df_us_left["us_sipm_idx"], df_us_left["d_sipm"], yerr=df_us_left["d_sipm_err"],
            fmt="+", color="red", markersize=6, capsize=0, label="left")
ax.errorbar(df_us_right["us_sipm_idx"], df_us_right["d_sipm"], yerr=df_us_right["d_sipm_err"],
            fmt="+", color="blue", markersize=6, capsize=0, label="right")

us_boundaries = [0, 160, 320, 480, 640, 800]
us_dividers = [80, 240, 400, 560, 720]
for b in us_boundaries:
    ax.axvline(b, color="lime", linewidth=2.0, linestyle="-")
for d in us_dividers:
    ax.axvline(d, color="magenta", linewidth=1.5, linestyle="--")

us_gel_pos = [(120, "G"), (200, "G"), (440, "G"), (520, "G"), (680, "G")]
for x_pos, text in us_gel_pos:
    ax.text(x_pos, 14.5, text, fontweight="bold", fontsize=13, ha="center", va="center")

ax.set_xlim(-10, 810)
ax.set_ylim(6, 16)
ax.set_xlabel("SiPM number", fontsize=14)
ax.set_ylabel(r"DS alignment parameter, $d_{\mathrm{SiPM}}$ [ns]", fontsize=14)
ax.set_title(f"DS alignment parameter, {{\mathrm{{SiPM}}}}$ as a function of SiPM number (Run {plot_run})", fontsize=15)
ax.grid(True, which="both", linestyle=":", alpha=0.5)
ax.legend(loc="upper right", frameon=True, framealpha=0.9, facecolor="white", edgecolor="black", fontsize=13)

plt.tight_layout()
plt.show()


---
## 6. Time-Walk Corrected 2D Distribution ($dt_{\mathrm{SiPM}}^{\mathrm{TW,ToF}}$ vs $\mathrm{QDC}_{\mathrm{SiPM}}$)

Visualizes the 2D distribution after subtracting the time-walk curve:
$$\Delta t^{\mathrm{TW}}(\mathrm{QDC}) = \frac{\alpha}{\beta \cdot \mathrm{QDC} - \mathrm{QDC}_0} + \gamma \cdot \mathrm{QDC}$$
This flattens the distribution across all QDC, directly illustrating the fine alignment parameter $d_{\mathrm{SiPM}}$ as the plateau level relative to zero (matching the dissertation demonstration plot).


In [ ]:
# Select channel and run to display
sample_run = 5408
target_ch = "24005_4"  # US Plane 5, Bar 5, SiPM 4 (Left side)
ref_type = "DS"

# Load channel data via loader utility
ch_data = load_channel_tw_root(sample_run, target_ch, ref_type=ref_type)
tw_p = ch_data["params"]
d_sipm = tw_p["alignment_d"]

# Plot 2D Time-Walk corrected distribution
fig, ax = plt.subplots(figsize=(10, 6))

masked_2d = np.ma.masked_where(ch_data["corr_2d"] <= 0, ch_data["corr_2d"])
mesh = ax.pcolormesh(ch_data["x_edges"], ch_data["y_edges"], masked_2d,
                     cmap="viridis",
                     norm=mcolors.LogNorm(vmin=1.0, vmax=max(10, np.nanmax(ch_data["corr_2d"]))),
                     shading="auto")
cbar = plt.colorbar(mesh, ax=ax, pad=0.02)
cbar.set_label("Counts (log scale)", fontsize=11)

ax.axhline(d_sipm, color="red", linestyle="--", linewidth=2.0)
ax.axhline(0.0, color="black", linestyle=":", linewidth=1.2, alpha=0.7)

ax.annotate("", xy=(165, 0.0), xytext=(165, d_sipm),
            arrowprops=dict(arrowstyle="<->", color="red", lw=2.0))
ax.text(145, 0.5 * d_sipm, rf"{{\mathrm{{SiPM}}}} = {d_sipm:.2f}$ ns",
        color="red", fontsize=11, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="red", alpha=0.9))

ax.set_xlabel(r"$\mathrm{QDC}_{\mathrm{SiPM}}$ [a.u.]", fontsize=12)
ax.set_ylabel(r"$\mathrm{dt}_{\mathrm{SiPM}}^{\mathrm{TW,ToF}}$ [ns]", fontsize=12)
ax.set_title(f"US plane 5, bar 5, SiPM 4 (left side) — $\mathrm{{dt}}_{{\mathrm{{SiPM}}}}^{{\mathrm{{TW,ToF}}}}$ vs $\mathrm{{QDC}}_{{\mathrm{{SiPM}}}}$ (Run {sample_run})", fontsize=12)
ax.set_xlim(0, 200)
ax.set_ylim(-2, 12)
ax.grid(True, linestyle=":", alpha=0.4)

plt.tight_layout()
plt.show()


---
## 7. Calibration Policy: Single Category Set vs. Run-by-Run Sets?

### Quantitative Assessment
To answer whether a single consolidated calibration set (such as `catII_timing_calibration.json`) can be used for all runs within a category or if run-dependent sets are required, we evaluate:
1. **Channel-Level Parameter Drift Over Runs:** Standard deviation of $d_{\mathrm{SiPM}}$ and $t_0$ across runs for each individual channel.
2. **Comparison with Intrinsic Time Resolution:** SiPM time resolution after TW correction is $\sim 250\text{--}350\,\mathrm{ps}$.
3. **Residual Distribution:** Channel-by-channel difference between individual run constants and the consolidated Category II calibration set.


In [ ]:
# Load Category II consolidated calibration
cat_file = os.path.join(fits_dir, "catII_timing_calibration.json")
has_cat_file = os.path.exists(cat_file)
if has_cat_file:
    with open(cat_file, "r") as f:
        cat_data = json.load(f)
    cat_al = cat_data.get("alignment", {})

# Compute per-channel drift (std across runs)
drift_records = []
for (ref, ch), group in df_channels.groupby(["ref", "channel"]):
    if len(group) < 5:
        continue
    d_std = group["d_sipm"].std()
    d_mean = group["d_sipm"].mean()
    t0_std = group["t0"].std()
    diff_to_cat = np.nan
    if has_cat_file and ref in cat_al and ch in cat_al[ref]:
        diff_to_cat = d_mean - cat_al[ref][ch][0]
    drift_records.append({
        "ref": ref,
        "channel": ch,
        "subsys": group["subsys"].iloc[0],
        "system": group["system"].iloc[0],
        "d_std": d_std,
        "t0_std": t0_std,
        "diff_to_cat": diff_to_cat,
    })

df_drift = pd.DataFrame(drift_records)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Run-to-run standard deviation of d_SiPM per channel
for ref, color in zip(["DS", "SF"], ["steelblue", "crimson"]):
    sub = df_drift[df_drift["ref"] == ref]["d_std"] * 1000.0  # in ps
    med_ps = sub.median()
    ax1.hist(sub, bins=40, range=(0, 400), alpha=0.6, color=color, edgecolor="black", linewidth=0.5,
             label=f"{ref} Ref (Median drift = {med_ps:.1f} ps)")
    ax1.axvline(med_ps, color=color, linestyle="--", linewidth=1.5)

ax1.axvline(260.0, color="black", linestyle=":", linewidth=2, label="SiPM Intrinsic Res ($\sim 260$ ps)")
ax1.set_xlabel(r"Channel-wise Run-to-Run Drift $\sigma_{\mathrm{run}}(d_{\mathrm{SiPM}})$ [ps]", fontsize=12)
ax1.set_ylabel("Number of Channels", fontsize=12)
ax1.set_title("Temporal Stability: Per-Channel Drift Across Runs")
ax1.grid(True, alpha=0.3)
ax1.legend(frameon=True, fontsize=10)

# Right plot: Difference between Run Average and Consolidated Category Set
if has_cat_file:
    for ref, color in zip(["DS", "SF"], ["steelblue", "crimson"]):
        sub = df_drift[df_drift["ref"] == ref]["diff_to_cat"].dropna() * 1000.0
        med_diff = sub.median()
        std_diff = sub.std()
        ax2.hist(sub, bins=40, range=(-300, 300), alpha=0.6, color=color, edgecolor="black", linewidth=0.5,
                 label=f"{ref} (Mean diff = {med_diff:.1f} ps, $\sigma$={std_diff:.1f} ps)")
    ax2.set_xlabel(r"{\mathrm{SiPM}}(\mathrm{Run\,Avg}) - d_{\mathrm{SiPM}}(\mathrm{Cat\,II})$ [ps]", fontsize=12)
    ax2.set_ylabel("Number of Channels", fontsize=12)
    ax2.set_title("Compatibility with Single Consolidated Category II Set")
    ax2.grid(True, alpha=0.3)
    ax2.legend(frameon=True, fontsize=10)

plt.tight_layout()
plt.show()

print("="*70)
print("CALIBRATION STABILITY SUMMARY:")
for ref in ["DS", "SF"]:
    sub = df_drift[df_drift["ref"] == ref]
    print(f"[{ref} Reference]")
    print(f"  -> Median channel drift across runs: {sub['d_std'].median()*1000.0:.1f} ps")
    print(f"  -> 90th percentile channel drift   : {sub['d_std'].quantile(0.90)*1000.0:.1f} ps")
print("="*70)


---
## 8. Fine Time Alignment ($d_{\mathrm{SiPM}}$) Extraction and $\pm 1\sigma$ Uncertainty Band

This plot illustrates how $d_{\mathrm{SiPM}}$ is extracted for a single SiPM channel (following Thesis Section 4.1.5 & Eq. 4.1.12):
1. **Left Panel (2D Distribution):** Shows $\mathrm{dt}_{\mathrm{SiPM}}^{\mathrm{TW,ToF}} = \Delta t - \Delta t^{\mathrm{TW}}(\mathrm{QDC})$ vs $\mathrm{QDC}$. The time-walk shape is removed, but the alignment offset $d_{\mathrm{SiPM}}$ remains. The red dashed line marks $d_{\mathrm{SiPM}}$, and the shaded red band shows the true $\pm 1\sigma$ distribution dispersion ($\sigma_{\mathrm{dist}} \approx 523\text{ ps}$) around the plateau.
2. **Right Panel (1D Projection):** The corresponding 1D projection of the plateau ($\{dt}_{\mathrm{SiPM}}^{\mathrm{TW,ToF}}$), highlighting $d_{\mathrm{SiPM}}$ and the $\pm 1\sigma$ truncated region from which $d_{\mathrm{SiPM}}$ and its uncertainty are extracted.


In [ ]:
# Load channel data via loader utility
data = load_channel_tw_root(sample_run=5408, target_ch="24005_4", ref_type="DS", y_min=6.0, y_max=14.0)
tw_params = data["params"]
d_sipm = tw_params["alignment_d"]
d_sipm_err = tw_params.get("alignment_d_err", 0.001)
sigma_dist = data["sigma_dist"]  # True 1-sigma distribution dispersion (~523 ps)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15.5, 6.2), gridspec_kw={"width_ratios": [2.3, 1.2]})

# 1. Left: 2D corrected distribution with true ±1 sigma (StdDev) band
masked_2d = np.ma.masked_where(data["corr_2d"] <= 0, data["corr_2d"])
mesh = ax1.pcolormesh(data["x_edges"], data["y_edges"], masked_2d, cmap="viridis",
                      norm=mcolors.LogNorm(vmin=1.0, vmax=max(10, np.nanmax(data["corr_2d"]))),
                      shading="auto")
cbar = plt.colorbar(mesh, ax=ax1, pad=0.02)
cbar.set_label("Counts (log scale)", fontsize=11)

ax1.axhline(d_sipm, color="red", linestyle="--", linewidth=2.0, label=rf"{{\mathrm{{SiPM}}}} = {d_sipm:.2f}\pm {d_sipm_err*1000:.1f}\\,\mathrm{{ps}}$")
ax1.axhspan(d_sipm - sigma_dist, d_sipm + sigma_dist, color="red", alpha=0.18, label=rf"$\pm 1\sigma$ Truncation Window ($\sigma_{{\mathrm{{dist}}}} = {sigma_dist*1000:.0f}\\,\mathrm{{ps}}$)")
ax1.axhline(d_sipm - sigma_dist, color="darkred", linestyle=":", linewidth=1.2)
ax1.axhline(d_sipm + sigma_dist, color="darkred", linestyle=":", linewidth=1.2)
ax1.axhline(0, color="black", linestyle=":", linewidth=1.0, alpha=0.7)

ax1.set_xlabel(r"$\mathrm{QDC}_{\mathrm{SiPM}}$ [a.u.]", fontsize=12)
ax1.set_ylabel(r"$\mathrm{dt}_{\mathrm{SiPM}}^{\mathrm{TW,ToF}}$ [ns]", fontsize=12)
ax1.set_title(f"Channel {data['channel']} (TW Corrected, Unaligned) with $\pm 1\sigma$ Truncation Window", fontsize=12.5)
ax1.set_xlim(0, 200)
ax1.set_ylim(6.0, 14.0)
ax1.grid(True, linestyle=":", alpha=0.4)
ax1.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.9, fontsize=10)

# 2. Right: 1D projection with shaded ±1 sigma truncated region
res_x = data["res_x"]
res_y = data["res_y"]
ax2.step(res_y, res_x, where="mid", color="steelblue", linewidth=1.5, label="1D Residual Projection")
ax2.fill_betweenx(res_x, 0, res_y, step="mid", color="steelblue", alpha=0.25)

# Highlight +/- 1 sigma band on 1D
mask_1sig = (res_x >= (d_sipm - sigma_dist)) & (res_x <= (d_sipm + sigma_dist))
ax2.fill_betweenx(res_x[mask_1sig], 0, res_y[mask_1sig], step="mid", color="red", alpha=0.35, label=rf"$\pm 1\sigma$ Truncated Region ({sigma_dist*1000:.0f} ps)")
ax2.axhline(d_sipm, color="red", linestyle="--", linewidth=2.0)
ax2.axhline(d_sipm - sigma_dist, color="darkred", linestyle=":", linewidth=1.2)
ax2.axhline(d_sipm + sigma_dist, color="darkred", linestyle=":", linewidth=1.2)

ax2.set_xlabel("Counts", fontsize=12)
ax2.set_ylabel(r"$\mathrm{dt}_{\mathrm{SiPM}}^{\mathrm{TW,ToF}}$ [ns]", fontsize=12)
ax2.set_title("1D Residual & Truncated Mean Region", fontsize=12.5)
ax2.set_ylim(6.0, 14.0)
ax2.grid(True, linestyle=":", alpha=0.4)
ax2.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.9, fontsize=9.5)

plt.tight_layout()
plt.show()
